## This one is for replacing DFFs with scanDFFs, hence inserting the scan chain

In [2]:
import os

In [3]:
benchmark_name = "custom_circuit_netlist"

In [4]:
## verilog preprocess:

file_path = benchmark_name+".v"
preVer = []
with open(file_path, "r") as file:
    for line in file:
        if "not" in line:
            outp = (line[line.index('(')+1:line.index(')')].split(','))[0].strip(" ")
            inp = (line[line.index('(')+1:line.index(')')].split(','))[1].strip(" ")
            line = line.replace("not ", "nand ").replace("NOT1_","NAND2_rep").replace(inp+");", inp+", "+inp+");")
            preVer.append(line)
        else:
            preVer.append(line)

directory_path = benchmark_name+"/Verilog/"
os.makedirs(directory_path, exist_ok=True)
Ver_path = os.path.join(directory_path+benchmark_name+".v")
with open(Ver_path, "w") as file:
    file.writelines(preVer)
file.close()


In [5]:
## Four input gates
class DFF:
    def __init__(self, name, numOfInp, clk, outputQ, inputD, gateNum, type):
        self.name = name
        self.numOfInp = numOfInp
        self.gateNum = gateNum
        self.clk = clk
        self.outputQ = outputQ
        self.inputD = inputD
        self.type = type
    def __str__(self):
        return f" gateNum = {self.gateNum}\n gateName = {self.name}\n D = {self.inputD}\n Q = {self.outputQ}\n gate Type = {self.type}"


In [6]:
directory_path = benchmark_name+"/Verilog/"
file_path = os.path.join(directory_path+benchmark_name+".v")
i = 0
j = 0
net = []
SCI_ver = []
newGate = False


with open(file_path, "r") as file:
    for line in file:  
        if ('dff' in line) and ('module' not in line):
            print(line)
            newGate = True
            numOfGateInput = -1 # means DFF
            # j=j+1
            currGateName = line.split()[1]
            print(currGateName)
            line = file.readline() ##first input
            C = (line.split('(')[1].split(')')[0])
            print ("C : "+str(C))
            line = file.readline() ##first input
            D = (line.split('(')[1].split(')')[0])
            print ("D : "+str(D))
            line = file.readline() ##first input
            Q = (line.split('(')[1].split(')')[0])
            print ("Q : "+str(Q))
            line = file.readline() ## );
            gateType = "dff"
            net.append(DFF(currGateName, -1, C, Q, D, j, gateType))

            SCI_ver.append(" dff "+currGateName + "(\n")
            SCI_ver.append(" \t.C(CK),\n \t.CE(1'b1),\n \t.CLR(rst),\n")
            SCI_ver.append(" \t.D("+D+"),\n")
            SCI_ver.append(" \t.NbarT(PbarS),\n \t.PRE(1'b0),\n")
            SCI_ver.append(" \t.Q("+Q+"),\n")
            if (j==0):
                SCI_ver.append(" \t.Si(Si),\n")
            else:
                SCI_ver.append(" \t.Si("+Q_prev+"),\n")
            SCI_ver.append(" \t.global_reset(1'b0)\n );\n\n")

            j=j+1
            Q_prev = Q

            if(j==179):
                SCI_ver.append(" assign So = "+Q+";\n\n")

        else:
            SCI_ver.append(line)
file.close()


directory_path = benchmark_name+"/SCI_verilog/"
os.makedirs(directory_path, exist_ok=True)
SCI_ver_file_path = os.path.join(directory_path+benchmark_name+"nnew.v")
with open(SCI_ver_file_path, "w") as file:
    file.writelines(SCI_ver)
file.close()

  dff DFF_0 (

DFF_0
C : clk
D : w1
Q : w5
  dff DFF_1 (

DFF_1
C : clk
D : w3
Q : w4


In [7]:
## systemC output
sysC_H = []
# for required_gate in required_gates:
#     sysC_H.append("#include \""+required_gate+"_X1.h\"\n")
sysC_H.append("#include \"Complex_NAgate_45.h\"\n")

sysC_H.append("SC_MODULE("+benchmark_name+")\n{\n")

sysC_H.append("\tsc_in <sc_logic> clk;\n")
sysC_H.append("\tsc_in <sc_logic> rst;\n")
sysC_H.append("\tsc_in <sc_logic> PbarS;\n")
sysC_H.append("\tsc_in <sc_logic> Si;\n")
sysC_H.append("\tsc_out <sc_logic> So;\n\n")
sysC_H.append("\tsc_signal <sc_logic> sc_logic_1_signal, sc_logic_0_signal;\n\n")


sysC_H.append("\tsc_in <sc_logic> ")
for PI_idx in range(len(PIs)-1):
    sysC_H.append(PIs[PI_idx]+", ")
sysC_H.append(PIs[len(PIs)-1]+";\n")

sysC_H.append("\tsc_out <sc_logic> ")
for PO_idx in range(len(POs)-1):
    sysC_H.append(POs[PO_idx]+", ")
sysC_H.append(POs[len(POs)-1]+";\n")

sysC_H.append("\tsc_signal <sc_logic> ")
for wire_idx in range(len(wires)-1):
    if((not(wires[wire_idx] in PIs)) and (not(wires[wire_idx] in POs))):
        sysC_H.append(wires[wire_idx]+", ")
sysC_H.append(wires[len(wires)-1]+";\n")

sysC_H.append("\tsc_in<sc_logic> endSim; \n")
sysC_H.append("\tsc_in<sc_logic> newTV; \n")
sysC_H.append("\tsc_uint<32> counter; \n")

sysC_H.append("\n\tint numOfGates;\n\tint totalObservedCombs;\n\tdouble GIC_Coverage;\n\n")


# sysC_H.append("\n\tint numOfGates;\n\tdouble t;\n\tdouble outLoad;\n\n")

i = 1
for g in net:
    # sysC_H.append("\t"+g.name+"_X1* "+g.name+"_Gate"+str(i)+";\n")
    sysC_H.append("\t"+g.type+"* "+g.name+"_Gate"+str(i)+";\n")
    i = i+1

sysC_H.append("\n\tSC_CTOR("+benchmark_name+")\n\t{\n\t\tnumOfGates = "+str(len(net))+";\n\n")

i = 1
for g in net:
    # sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+" = new "+g.name+"_X1(\""+g.name+"_instance"+str(g.gateNum)+"\");\n")
    sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+" = new "+g.type+"(\""+g.name+"_instance"+str(g.gateNum)+"\");\n")
    if (g.numOfInp == -1):
        sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->C"+"("+g.clk+");\n")
        sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->CE"+"(sc_logic_1_signal);\n")
        sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->CLR"+"(rst);\n")
        sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->D"+"("+g.inputD+");\n")
        sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->NbarT"+"(PbarS);\n")
        sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->PRE"+"(sc_logic_0_signal);\n")
        sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->Si"+"();\n")
        sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->global_reset"+"(sc_logic_0_signal);\n")

    if (g.numOfInp == 1):
        sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->A"+"("+g.inputA1+");\n")

    if (g.numOfInp == 2):
        if (g.name.startswith("XOR")): ## XOR has only 2
            sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->A"+"("+g.inputA1+");\n")
            sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->B"+"("+g.inputA2+");\n")
        else:    
            sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->A1"+"("+g.inputA1+");\n")
            sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->A2"+"("+g.inputA2+");\n")
    if (g.numOfInp == 3):
        sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->A1"+"("+g.inputA1+");\n")
        sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->A2"+"("+g.inputA2+");\n")
        sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->A3"+"("+g.inputA3+");\n")
    if (g.numOfInp == 4):
        sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->A1"+"("+g.inputA1+");\n")
        sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->A2"+"("+g.inputA2+");\n")
        sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->A3"+"("+g.inputA3+");\n")
        sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->A4"+"("+g.inputA4+");\n")
    if(g.numOfInp == -1):
        sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->Q"+"("+g.outputQ+");\n\n")
    else:
        sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->ZN"+"("+g.outputZN+");\n\n")
    # if (g.outputZN in POs):
    #     sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->load_c"+" = outLoad;\n")
    # else:
    #     sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->load_c = 0;\n")
    # sysC_H.append("\t\t"+g.name+"_Gate"+str(i)+"->aged_time = t;\n\n")
    i = i+1

sysC_H.append("\t\tcout << \"all gates are instantiated \" << numOfGates << \"\\n\";\n")

sysC_H.append("\t\tSC_METHOD(sc_logic_signal_assignments);\n")

sysC_H.append("\t\tSC_THREAD(assignments);\n")
# //sensitivity
j=0
unique_sens_list = list(set(assign_right))
sysC_H.append("\t\tsensitive")
for j in range(len(unique_sens_list)):
    sysC_H.append(" << "+unique_sens_list[j])
sysC_H.append(";\n")
sysC_H.append("\t\tSC_METHOD(GIC_Coverage_Calculator);\n")
sysC_H.append("\t\tsensitive << endSim<< newTV;\n\n\t}\n")

sysC_H.append("\tvoid ini();\n\tvoid assignments();\n\tvoid GIC_Coverage_Calculator();\n")

sysC_H.append("\tvoid sc_logic_signal_assignments(){\n\t\tsc_logic_1_signal.write(SC_LOGIC_1);\n\t\tsc_logic_0_signal.write(SC_LOGIC_0);\n\t}\n")


# j = 1
# for required_gate_idx in range(len(required_gates)):
#     sysC_H.append("\tvoid notifyAlfaCalc"+required_gates[required_gate_idx]+"(")
#     sysC_H.append(required_gates[required_gate_idx]+"_X1 *gate"+");\n")
#     # sysC_H.append(required_gates[len(required_gates)-1]+"_X1 *gate"+str(j)+");\n};\n")
#     j=j+1
sysC_H.append("\n};\n")


directory_path = benchmark_name+"/SystemC/"
os.makedirs(directory_path, exist_ok=True)
sysC_output_file_path = os.path.join(directory_path+"netlist.h")
with open(sysC_output_file_path, "w") as file:
    file.writelines(sysC_H)
file.close()

NameError: name 'PIs' is not defined

In [8]:
sysC_C = []
sysC_C.append("#include \"netlist.h\"\n#include <cmath>\n")
sysC_C.append("std::ofstream GIC_logFile(\"GIC_logFile.txt\");\n\n")
sysC_C.append("void "+benchmark_name+"::assignments()\n{\n\twhile (true)\n\t{\n")

m=0
for m in range(len(assign_left)):
    sysC_C.append("\t\t"+assign_left[m]+".write("+assign_right[m]+");\n")

sysC_C.append("\n\t\twait();\n\t}\n}\n\n")

sysC_C.append("void "+benchmark_name+"::GIC_Coverage_Calculator()\n{\n")
sysC_C.append("\ttotalObservedCombs =\n")
n=0
for n in range(len(net)):
    if n==len(net)-1:
        sysC_C.append("\t"+net[n].name+"_Gate"+str(n+1)+"->numOfObservedCombs;\n")
    else:
        sysC_C.append("\t"+net[n].name+"_Gate"+str(n+1)+"->numOfObservedCombs +\n")

sysC_C.append("\tcout << \"END OF SIM=> Total Observed Combs = \" << totalObservedCombs << \"\\n\";\n")
sysC_C.append("\tcout << \"GIC Coverage = \" << GIC_Coverage << \"\\n\";\n\n}")



directory_path = benchmark_name+"/SystemC/"
os.makedirs(directory_path, exist_ok=True)
sysC_C_output_file_path = os.path.join(directory_path+"netlist.cpp")
with open(sysC_C_output_file_path, "w") as file:
    file.writelines(sysC_C)
file.close()

NameError: name 'assign_left' is not defined

In [9]:

sysC_TB_H = []
sysC_TB_H.append("#include \"netlist.h\"\n#include <fstream>\n\nSC_MODULE("+benchmark_name+"_TB)\n{\n\n")
sysC_TB_H.append("\tsc_signal <sc_logic> ")
for PI_idx in range(len(PIs)-1):
    sysC_TB_H.append("testData"+str(PI_idx+1)+", ")
sysC_TB_H.append("testData"+str(len(PIs))+";\n")

sysC_TB_H.append("\tsc_signal <sc_logic> ")
for PO_idx in range(len(POs)-1):
    sysC_TB_H.append("testRes"+str(PO_idx+1)+", ")
sysC_TB_H.append("testRes"+str(len(POs))+";\n")

sysC_TB_H.append("\n\tsc_signal<sc_logic> reset, clock, end, newTV;\n\t"+benchmark_name+"* UUT;\n\n")
sysC_TB_H.append("\tstd::vector<std::string> testVecs;\n")

sysC_TB_H.append("\tSC_CTOR("+benchmark_name+"_TB)\n\t{\n\n\t\ttestVecs = read_testPtr (\"testPatterns.txt\");\n\t\tUUT = new "+benchmark_name+"(\""+benchmark_name+"_instance\");\n")
p=1
for PI in PIs:
    sysC_TB_H.append("\t\tUUT->"+PI.strip(" ")+"(testData"+str(p)+");\n")
    p=p+1

o=1
for PO in POs:
    sysC_TB_H.append("\t\tUUT->"+PO.strip(" ")+"(testRes"+str(o)+");\n")
    o=o+1

sysC_TB_H.append("\t\tUUT->endSim(end);\n")
sysC_TB_H.append("\t\tUUT->newTV(newTV);\n")
sysC_TB_H.append("\n\t\tSC_THREAD(testPtr);\n\t\tSC_THREAD(endOfSim);\n\t}\n")

sysC_TB_H.append("\n\tvoid endOfSim();\n\tvoid testPtr();\n\tstd::vector<std::string> read_testPtr(std::string filename);\n")
sysC_TB_H.append("};\n")




directory_path = benchmark_name+"/SystemC/"
os.makedirs(directory_path, exist_ok=True)
sysC_TB_H_output_file_path = os.path.join(directory_path+"TB.h")
with open(sysC_TB_H_output_file_path, "w") as file:
    file.writelines(sysC_TB_H)
file.close()

NameError: name 'PIs' is not defined

In [ ]:
sysC_TB_C = []
sysC_TB_C.append("#include \"TB.h\"\n\n")
sysC_TB_C.append("void "+benchmark_name+"_TB::testPtr()\n{\n\twhile (true)\n\t{\n")

m=1
for PI in PIs:
    # sysC_TB_C.append("void powerGatesNetlistTB::testData"+str(m)+"Waveform()\n{\n\twhile (true)\n{\n")
    sysC_TB_C.append("\t\ttestData"+str(m)+".write(SC_LOGIC_0);\n")
    m=m+1

sysC_TB_C.append("\t\twait(1000, SC_NS);\n")
sysC_TB_C.append("\t\tfor (int testVecidx = 0; testVecidx < testVecs.size(); testVecidx++)\n\t\t\t{\n")
s=0
for s in range(len(PIs)):
    sysC_TB_C.append("\t\t\tif (testVecs[testVecidx]["+str(s)+"] == '0')\n\t\t\t\ttestData"+str(s+1)+".write(SC_LOGIC_0);\n\t\t\telse if (testVecs[testVecidx]["+str(s)+"] == '1')\n\t\t\t\ttestData"+str(s+1)+".write(SC_LOGIC_1);\n\n")
sysC_TB_C.append("\t\twait(15000, SC_NS);\n\t\tnewTV.write(SC_LOGIC_1);\n\t\twait(0, SC_NS);\n\t\t}\n\t\twait();\n\t}\n}\n")

sysC_TB_C.append("std::vector<std::string> "+benchmark_name+"_TB::read_testPtr(std::string filename)\n{\n")
sysC_TB_C.append("\tstd::vector<std::string> selected_testVec;\n\tstd::ifstream file(filename);\n\tif (!file.is_open()) {\n\t\tstd::cerr << \"Error opening file!\" << std::endl;\t\n\t}\n\tstd::string line;\n\tstd::vector<std::string> lines;\n\twhile (std::getline (file, line))\n\t{\n\t\tlines.push_back(line);\n\t}\n\treturn lines;\n}")

sysC_TB_C.append("\nvoid "+benchmark_name+"_TB::endOfSim()\n{\n\twhile (true)\n{\n\t\tend.write(SC_LOGIC_0);\n\t\twait(3900000, SC_NS);\n\t\tend.write(SC_LOGIC_1);\n\t\twait(15000, SC_NS);\n\t\twait();\n\t}\n}")

directory_path = benchmark_name+"/SystemC/"
os.makedirs(directory_path, exist_ok=True)
sysC_TB_C_output_file_path = os.path.join(directory_path+"TB.cpp") 
with open(sysC_TB_C_output_file_path, "w") as file:
    file.writelines(sysC_TB_C)
file.close()

NameError: name 'PIs' is not defined

In [10]:
sysC_sim_C = []
sysC_sim_C.append("#include \"TB.h\"\n#include <iostream>\n#include <fstream> \n\nint sc_main(int argc, char** argv)\n{\n\t"+benchmark_name+"_TB* TOP = new "+benchmark_name+"_TB(\"netlistSimulationTB_instance\");")
sysC_sim_C.append("\n\tsc_start(400000, SC_NS);\n\treturn 0;\n}")

directory_path = benchmark_name+"/SystemC/"
os.makedirs(directory_path, exist_ok=True)
sysC_sim_C_output_file_path = os.path.join(directory_path+"simulation.cpp") 
with open(sysC_sim_C_output_file_path, "w") as file:
    file.writelines(sysC_sim_C)
file.close()